# Day 060 — Solution: Caching & Performance

In [ ]:
_CACHING_API_SRC = '"""caching_api.py — Day 060: response caching for AI apps.\n\nRun:  uvicorn caching_api:app --reload\nDocs: http://localhost:8000/docs\n"""\nimport os\nimport time\nfrom datetime import datetime\n\nimport ollama\nfrom fastapi import FastAPI\nfrom pydantic import BaseModel, Field\n\nMODEL     = os.environ.get("MODEL", "llama3.2")\nCACHE_TTL = float(os.environ.get("CACHE_TTL_SECONDS", "300"))\nAPP_VER   = "1.0.0"\n\n\nclass SimpleCache:\n    """In-memory key-value cache with per-entry TTL."""\n\n    def __init__(self):\n        self._store: dict = {}   # key -> (value, expires_at)\n\n    def set(self, key: str, value, ttl: float = 60.0) -> None:\n        self._store[key] = (value, time.monotonic() + ttl)\n\n    def get(self, key: str):\n        entry = self._store.get(key)\n        if entry is None:\n            return None\n        value, expires_at = entry\n        if time.monotonic() > expires_at:\n            del self._store[key]\n            return None\n        return value\n\n    def has(self, key: str) -> bool:\n        return self.get(key) is not None\n\n    def delete(self, key: str) -> None:\n        self._store.pop(key, None)\n\n    def clear(self) -> int:\n        n = len(self._store)\n        self._store.clear()\n        return n\n\n    def __len__(self) -> int:\n        now = time.monotonic()\n        return sum(1 for _, exp in self._store.values() if now <= exp)\n\n\nclass AskRequest(BaseModel):\n    prompt: str = Field(min_length=1)\n\n\ndef build_api(process_fn=None) -> FastAPI:\n    """Build the caching API.\n\n    process_fn: optional callable(prompt: str) -> str for testing.\n    """\n    app   = FastAPI(title="Caching API", version=APP_VER)\n    cache = SimpleCache()\n    stats = {"hits": 0, "misses": 0}\n\n    @app.get("/health")\n    def health():\n        return {"status": "ok",\n                "timestamp": datetime.utcnow().isoformat() + "Z",\n                "version": APP_VER}\n\n    @app.post("/ask")\n    def ask(req: AskRequest):\n        cached = cache.get(req.prompt)\n        if cached is not None:\n            stats["hits"] += 1\n            return {"answer": cached, "cache_hit": True}\n\n        stats["misses"] += 1\n        if process_fn is not None:\n            answer = process_fn(req.prompt)\n        else:\n            resp = ollama.chat(\n                model=MODEL,\n                messages=[{"role": "user", "content": req.prompt}],\n            )\n            answer = resp["message"]["content"]\n        cache.set(req.prompt, answer, ttl=CACHE_TTL)\n        return {"answer": answer, "cache_hit": False}\n\n    @app.get("/cache/stats")\n    def cache_stats():\n        return {"hits": stats["hits"], "misses": stats["misses"],\n                "size": len(cache)}\n\n    @app.delete("/cache")\n    def clear_cache():\n        n = cache.clear()\n        stats["hits"] = 0\n        stats["misses"] = 0\n        return {"cleared": n}\n\n    return app\n\n\napp = build_api()\n\nif __name__ == "__main__":\n    import uvicorn\n    PORT = int(os.environ.get("PORT", "8000"))\n    uvicorn.run(app, host="0.0.0.0", port=PORT)\n'
from pathlib import Path
Path('caching_api.py').write_text(_CACHING_API_SRC)
print('caching_api.py written.')

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel, Field
from starlette.testclient import TestClient

# ── inline test app (no Ollama) ───────────────────────────────────────────────
call_count = {"n": 0}

def fake_process(prompt: str) -> str:
    call_count["n"] += 1
    return f"Answer to: {prompt}"

import time

class _SC:
    def __init__(self):
        self._s: dict = {}
    def set(self, k, v, ttl=60.0):
        self._s[k] = (v, time.monotonic() + ttl)
    def get(self, k):
        e = self._s.get(k)
        if not e: return None
        v, exp = e
        if time.monotonic() > exp:
            del self._s[k]; return None
        return v
    def has(self, k): return self.get(k) is not None
    def clear(self):
        n = len(self._s); self._s.clear(); return n
    def __len__(self):
        now = time.monotonic()
        return sum(1 for _, exp in self._s.values() if now <= exp)

cache = _SC()
stats = {"hits": 0, "misses": 0}

class _AskReq(BaseModel):
    prompt: str = Field(min_length=1)

test_app = FastAPI()

@test_app.get("/health")
def _health():
    from datetime import datetime
    return {"status": "ok", "timestamp": datetime.utcnow().isoformat() + "Z",
            "version": "1.0.0"}

@test_app.post("/ask")
def _ask(req: _AskReq):
    cached = cache.get(req.prompt)
    if cached is not None:
        stats["hits"] += 1
        return {"answer": cached, "cache_hit": True}
    stats["misses"] += 1
    answer = fake_process(req.prompt)
    cache.set(req.prompt, answer, ttl=60.0)
    return {"answer": answer, "cache_hit": False}

@test_app.get("/cache/stats")
def _stats():
    return {"hits": stats["hits"], "misses": stats["misses"], "size": len(cache)}

@test_app.delete("/cache")
def _clear():
    n = cache.clear(); stats["hits"] = 0; stats["misses"] = 0
    return {"cleared": n}

client = TestClient(test_app, raise_server_exceptions=False)

# /health
r = client.get("/health")
assert r.status_code == 200 and r.json()["status"] == "ok"
print("\u2705 /health works")

# first ask → miss
r1 = client.post("/ask", json={"prompt": "What is caching?"})
assert r1.status_code == 200
b1 = r1.json()
assert b1["cache_hit"] is False and "Answer to:" in b1["answer"]
assert call_count["n"] == 1
print("\u2705 first POST /ask calls process_fn (miss)")

# same prompt → hit
r2 = client.post("/ask", json={"prompt": "What is caching?"})
b2 = r2.json()
assert b2["cache_hit"] is True and call_count["n"] == 1
print("\u2705 repeated prompt returns cache_hit=True (fn not called again)")

# stats
rs = client.get("/cache/stats")
s = rs.json()
assert s["hits"] == 1 and s["misses"] == 1 and s["size"] == 1
print("\u2705 /cache/stats reports correct hits/misses/size")

# DELETE /cache
rd = client.delete("/cache")
assert rd.json()["cleared"] == 1
assert client.get("/cache/stats").json()["size"] == 0
print("\u2705 DELETE /cache clears everything")

# empty prompt → 422
r3 = client.post("/ask", json={"prompt": ""})
assert r3.status_code == 422
print("\u2705 empty prompt \u2192 422")

print("\nDay 060 \u2014 Caching & Performance complete! \U0001f389")
